In [36]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
import torch
import json
import transformers
import random
from datasets import load_dataset
import copy

In [37]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", cache_dir="/project/phan/nk569")
model

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-32B")

In [3]:
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen3-32B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=False, lstrip=

In [4]:
tokenizer.save_pretrained("./tokenizer")

('./tokenizer/tokenizer_config.json',
 './tokenizer/chat_template.jinja',
 './tokenizer/tokenizer.json')

### After this, you will have the folder of the original tokenizer of the model in "./tokenizer"
### Now get the dict for the vocab

In [5]:
total = len(tokenizer)

In [6]:
keys = range(total)
values = list(range(total))
random.shuffle(values)

In [7]:
finalDict = dict(zip(keys,values))

In [8]:
with open("./tokenizer/dict_Qwen3-32B.json", "w") as json_file:
    json.dump(finalDict, json_file)

In [9]:
vocab = tokenizer.get_vocab()

In [10]:
id_to_token = {v: k for k, v in vocab.items()}

# Create a new vocabulary list based on the custom mapping
new_vocab = [None] * len(vocab)
for old_index, new_index in finalDict.items():
    token = id_to_token[int(old_index)]
    new_vocab[new_index] = token

# Create a new token-to-id dictionary
new_token_to_id = {token: idx for idx, token in enumerate(new_vocab)}

### After getting the dict and index shift

In [11]:
!mv ./tokenizer/tokenizer_config.json ./tokenizer/tokenizer_config_temp.json
!mv ./tokenizer/tokenizer.json ./tokenizer/tokenizer_config.json

In [12]:
from transformers.models.auto.tokenization_auto import get_tokenizer_config

In [13]:
tokenizer_config = get_tokenizer_config("./tokenizer")

In [14]:
tokenizer_config_temp = tokenizer_config

In [15]:

for key in tokenizer_config.keys():
    print(key)

version
truncation
padding
added_tokens
normalizer
pre_tokenizer
post_processor
decoder
model
_commit_hash


In [16]:
for key in tokenizer_config['model']['vocab'].keys():
    print(key)
    print(tokenizer_config['model']['vocab'][key])
    break

!
0


In [17]:
for i in tokenizer_config['added_tokens']:
    i['id'] = new_token_to_id[i['content']]

In [18]:
tokenizer_config.pop("_commit_hash")

In [19]:
tokenizer_config["model"]["vocab"] = new_token_to_id

In [20]:
with open('./tokenizer/tokenizer.json', 'w' ) as f:
    json.dump(tokenizer_config, f,ensure_ascii=False, indent=2)

In [21]:
!rm -r ./tokenizer/tokenizer_config.json
!mv ./tokenizer/tokenizer_config_temp.json ./tokenizer/tokenizer_config.json

In [22]:
for key in tokenizer_config.keys():
    print(key)

version
truncation
padding
added_tokens
normalizer
pre_tokenizer
post_processor
decoder
model


In [23]:
tokenizer_config_new = get_tokenizer_config("./tokenizer")

In [24]:
for key in tokenizer_config_new.keys():
    print(key)

add_prefix_space
backend
bos_token
clean_up_tokenization_spaces
eos_token
errors
extra_special_tokens
is_local
model_max_length
pad_token
split_special_tokens
tokenizer_class
unk_token
_commit_hash


In [25]:
tokenizer_config_new.pop("_commit_hash")

In [26]:
added_tokens_decoder = {}
for i in tokenizer_config['added_tokens']:
    index = i['id']
    i.pop('id')
    added_tokens_decoder[index] = i

In [27]:
added_tokens_decoder

{128283: {'content': '<|endoftext|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 101051: {'content': '<|im_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 107667: {'content': '<|im_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 61192: {'content': '<|object_ref_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 19807: {'content': '<|object_ref_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 135502: {'content': '<|box_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 53094: {'content': '<|box_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': Tru

In [28]:
content_to_index = {
    v["content"]: k
    for k, v in added_tokens_decoder.items()
}

In [29]:
tokenizer_config_new['added_tokens_decoder'] =  added_tokens_decoder

In [30]:
tokenizer_config_new['added_tokens_decoder']

{128283: {'content': '<|endoftext|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 101051: {'content': '<|im_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 107667: {'content': '<|im_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 61192: {'content': '<|object_ref_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 19807: {'content': '<|object_ref_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 135502: {'content': '<|box_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 53094: {'content': '<|box_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': Tru

In [31]:
tokenizer_config['added_tokens']

[{'content': '<|endoftext|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 {'content': '<|im_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 {'content': '<|im_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 {'content': '<|object_ref_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 {'content': '<|object_ref_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 {'content': '<|box_start|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 {'content': '<|box_end|>',
  'single_word': False,
  'lstrip': False,
  'rstrip': False,
  'normalized': False,
  'special': True},
 {'content': '<|quad_start|>',
  'single_word': F

In [32]:
added_tokens_dict = {}
for token_info in tokenizer_config['added_tokens']:
    token_content = token_info['content']
    added_tokens_dict[token_content] = content_to_index[token_content]

In [33]:
with open('./tokenizer/tokenizer_config.json', 'w' ) as f:
        json.dump(tokenizer_config_new, f,ensure_ascii=False, indent=2)

with open('./tokenizer/added_tokens.json', 'w', encoding='utf-8') as f:
    json.dump(added_tokens_dict, f, ensure_ascii=False, indent=2)

In [34]:
new_tok = AutoTokenizer.from_pretrained("./tokenizer")

In [35]:
new_tok

Qwen2Tokenizer(name_or_path='./tokenizer', vocab_size=151669, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	5870: AddedToken("<|vision_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	9069: AddedToken("<|vision_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	11978: AddedToken("<|fim_pad|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	19807: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	24866: AddedToken("<|fim_suffix|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	32211: AddedToken("<|vision_pad|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	52944: AddedToken("<|quad_start|>", rstrip=False, lstrip=False,